In [ ]:
import os
import shutil
from collections import defaultdict
from pathlib import Path

import pandas as pd
import supervision as sv
import yaml
from IPython.display import Image
from ultralytics import YOLO
from ultralytics.data import build

from smktscnr.components.balancer import YOLOWeightedDataset

os.chdir("..")

#### Prepare Dataset

In [ ]:
expt_name = "SupermarketScanner-Fine-Tune"

pth_data = Path("datasets")
pth_expt = pth_data / expt_name
pth_ds_dummy = pth_data / "online_images"
pth_ds_prod = pth_data / "checkout_kiosk"

pth_expt.mkdir(parents=True, exist_ok=True)

In [ ]:
labels = [
    "blueberry",
    "bread",
    "chicken",
    "egg",
    "juice",
    "melon",
    "sushi",
    "watermelon",
]
names = {k:v for k, v in enumerate(labels)}

data = dict(
    names=names,
    nc=len(names),
    train=os.path.join("train", "images"),
    val=os.path.join("dev", "images"),
    test=os.path.join("test", "images"),
)

with open(pth_expt / "data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False)

In [ ]:
ds_dummy = sv.DetectionDataset.from_yolo(
    images_directory_path=pth_ds_dummy / "images",
    annotations_directory_path=pth_ds_dummy / "labels",
    data_yaml_path=pth_expt / "data.yaml",
)

ds_prod = sv.DetectionDataset.from_yolo(
    images_directory_path=pth_ds_prod / "images",
    annotations_directory_path=pth_ds_prod / "labels",
    data_yaml_path=pth_expt / "data.yaml",
)

In [ ]:
ds_train_dev, ds_test = ds_prod.split(
    split_ratio=0.7,
    random_state=42,
    shuffle=True,
)

ds_train, ds_dev = ds_train_dev.split(
    split_ratio=0.3,
    random_state=42,
    shuffle=True,
)

ds_train = sv.DetectionDataset.merge([ds_train, ds_dummy])

In [ ]:
for split_set, ds in {"train": ds_train, "dev": ds_dev, "test": ds_test}.items():
    ds.as_yolo(
        images_directory_path=pth_expt / split_set / "images",
        annotations_directory_path=pth_expt / split_set / "labels",
    )

In [ ]:
subset_cnt = {"train": defaultdict(int), "dev": defaultdict(int), "test": defaultdict(int)}

for subset in subset_cnt:
    ds = sv.DetectionDataset.from_yolo(
        images_directory_path=pth_expt / subset / "images",
        annotations_directory_path=pth_expt / subset / "labels",
        data_yaml_path=pth_expt / "data.yaml",
    )
    
    for img in ds.annotations.values():
        for cid in img.class_id:
            subset_cnt[subset][data["names"][int(cid)]] += 1

df = pd.DataFrame(subset_cnt).fillna(0).astype(int).sort_index()
pd.concat([df, df.sum().to_frame("Total").T])

#### Fine Tune

In [ ]:
model = YOLO(Path("checkpoints", "smktscnr.pt"))
build.YOLODataset = YOLOWeightedDataset

_ = model.train(
    data=Path("datasets", "SupermarketScanner-Fine-Tune", "data.yaml"),
    # epochs=64,
    epochs=1,
    patience=8,
    name="smktscnr_ft",
    lr0=0.0001,
)

In [ ]:
Image(Path("runs", "segment", "smktscnr_ft", "results.png"))

In [ ]:
Image(Path("runs", "segment", "smktscnr_ft", "confusion_matrix.png"))

In [ ]:
model.val(
    split="test",
    name="smktscnr_ft_val",
)

#### Export Model

In [ ]:
model.export(
    format="onnx",
    imgsz=1_024,
    optimize=True,
    half=True,
)

#### Centralise Model

In [ ]:
shutil.copy2(
    Path("runs", "segment", "smktscnr_ft", "weights", "best.pt"),
    Path("checkpoints", "smktscnr_ft.pt"),
)

shutil.copy2(
    Path("runs", "segment", "smktscnr_ft", "weights", "best.onnx"),
    Path("checkpoints", "smktscnr_ft.onnx"),
)